# Free Colab vLLM GPU Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrrahman1517/nemotron-vl-embed-stack/blob/codex-gemma4-max-vs-vllm-colab/vllm_free_colab_benchmark.ipynb)

This notebook is for a **free Colab GPU** workflow when you want a quick, usable **vLLM-only** throughput measurement on an open model.

What it does:

- detects the current NVIDIA GPU
- chooses a small open model that should fit typical free Colab runtimes
- launches a local `vllm serve` endpoint
- runs a small concurrent benchmark against `/v1/chat/completions`
- reports request throughput, token throughput, and latency percentiles

What it does **not** do:

- it does **not** compare against MAX
- it does **not** verify the April 2, 2026 Modular Gemma 4 claim
- it does **not** use TPU


## Source Notes

Primary sources:

- Colab overview: [Google Colab](https://colab.google/)
- Colab GPU note that the assigned NVIDIA GPU may vary across sessions: [RAPIDS cuDF on Colab](https://colab.google/articles/cudf)
- vLLM TPU quickstart showing that TPU support is a separate `vllm-tpu` flow on a **Google Cloud TPU VM**, not this Colab GPU notebook: [vLLM TPU Quickstart](https://docs.vllm.ai/projects/tpu/en/latest/getting_started/quickstart/)

Practical interpretation:

- free Colab can give you a GPU, but the exact GPU varies by session
- this notebook is designed around the common free-tier GPU case, especially **T4-class** sessions
- if you land on **TPU v5e-1**, this is the wrong notebook because vLLM TPU uses a different stack and MAX is not part of that flow


## Recommended Use

- On a **T4 / 16 GiB-class** runtime, keep the default model and benchmark settings
- On a **larger free GPU** if you happen to get one, the notebook will automatically choose a slightly larger open model
- If Colab gives you **no GPU**, reconnect or change the runtime type to GPU if that option is available

This notebook favors **reliability on free Colab** over squeezing out the absolute highest possible throughput.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv", "requests", "pandas", "matplotlib"],
    check=True,
)


In [ ]:
%%writefile vllm_free_colab_benchmark_helper.py
from __future__ import annotations

import json
import shlex
import signal
import statistics
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any

try:
    import requests
except ImportError:  # pragma: no cover - installed in Colab
    requests = None


@dataclass
class GPUInfo:
    name: str
    memory_gb: float
    driver_version: str
    compute_capability: str | None = None


@dataclass
class ManagedProcess:
    name: str
    command: list[str]
    process: subprocess.Popen[str]
    log_path: Path


def capture(cmd: list[str]) -> str:
    print("$", shlex.join(cmd))
    completed = subprocess.run(
        cmd,
        check=True,
        text=True,
        capture_output=True,
    )
    return completed.stdout.strip()


def detect_gpu() -> GPUInfo | None:
    try:
        output = capture(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version,compute_cap",
                "--format=csv,noheader,nounits",
            ]
        )
    except Exception:
        return None

    first_line = output.splitlines()[0].strip()
    parts = [part.strip() for part in first_line.split(",")]
    if len(parts) < 3:
        return None

    return GPUInfo(
        name=parts[0],
        memory_gb=round(float(parts[1]) / 1024.0, 2),
        driver_version=parts[2],
        compute_capability=parts[3] if len(parts) >= 4 else None,
    )


def choose_free_colab_model(
    gpu: GPUInfo | None,
    force_model: str | None = None,
) -> tuple[str, str]:
    if force_model:
        return force_model, "Using a user-forced model override."

    if gpu is None:
        return (
            "Qwen/Qwen2.5-0.5B-Instruct",
            "GPU detection failed, so the notebook is using a very small open model to keep the vLLM benchmark runnable.",
        )

    if gpu.memory_gb >= 20:
        return (
            "Qwen/Qwen2.5-3B-Instruct",
            "This runtime has enough memory for a slightly larger open model while still being practical for Colab benchmarking.",
        )

    return (
        "Qwen/Qwen2.5-1.5B-Instruct",
        "This model is small enough for typical free Colab T4 runtimes while still giving a useful vLLM throughput signal.",
    )


def start_logged_process(
    name: str,
    command: list[str],
    *,
    log_dir: str | Path,
    env: dict[str, str] | None = None,
) -> ManagedProcess:
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{name}.log"
    log_file = log_path.open("w", encoding="utf-8")

    print("$", shlex.join(command))
    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    return ManagedProcess(name=name, command=command, process=process, log_path=log_path)


def tail_log(log_path: str | Path, *, lines: int = 80) -> str:
    log_path = Path(log_path)
    if not log_path.exists():
        return f"{log_path} does not exist."
    return "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


def ensure_server_ready(
    base_url: str,
    handle: ManagedProcess,
    *,
    timeout_s: int = 1800,
    log_interval_s: int = 30,
) -> None:
    if requests is None:
        raise RuntimeError("requests is required for readiness checks.")

    deadline = time.time() + timeout_s
    next_log_time = time.time() + log_interval_s
    last_error = None

    while time.time() < deadline:
        if handle.process.poll() is not None:
            raise RuntimeError(
                f"{handle.name} exited before becoming ready. "
                f"Last log lines:\n{tail_log(handle.log_path, lines=120)}"
            )

        try:
            response = requests.get(f"{base_url.rstrip('/')}/v1/models", timeout=10)
            if response.ok:
                print(f"Server ready at {base_url}")
                return
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:  # pragma: no cover - Colab specific
            last_error = str(exc)

        if time.time() >= next_log_time:
            print(f"Still waiting for {handle.name} at {base_url}")
            print(tail_log(handle.log_path, lines=80))
            next_log_time = time.time() + log_interval_s

        time.sleep(5)

    raise TimeoutError(
        f"Server at {base_url} did not become ready within {timeout_s}s. "
        f"Last error: {last_error}\nLast log lines:\n{tail_log(handle.log_path, lines=120)}"
    )


def warmup_chat(base_url: str, model: str) -> dict[str, Any]:
    if requests is None:
        raise RuntimeError("requests is required for warmup requests.")

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": "Reply with the single word ready."}],
        "temperature": 0,
        "max_tokens": 8,
    }
    response = requests.post(
        f"{base_url.rstrip('/')}/v1/chat/completions",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()
    return response.json()


def percentile(values: list[float], ratio: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, int(round((len(ordered) - 1) * ratio))))
    return ordered[index]


def benchmark_chat_completions(
    *,
    base_url: str,
    model: str,
    num_requests: int = 24,
    concurrency: int = 2,
    max_tokens: int = 96,
    timeout_s: int = 300,
) -> dict[str, Any]:
    if requests is None:
        raise RuntimeError("requests is required for benchmarking.")

    prompt = (
        "Explain in one short paragraph how tensor cores help transformer inference. "
        "Keep the answer factual and concise."
    )

    def one_request(request_id: int) -> dict[str, Any]:
        payload = {
            "model": model,
            "messages": [{"role": "user", "content": f"{prompt} Request {request_id}."}],
            "temperature": 0,
            "max_tokens": max_tokens,
        }
        started = time.perf_counter()
        response = requests.post(
            f"{base_url.rstrip('/')}/v1/chat/completions",
            json=payload,
            timeout=timeout_s,
        )
        latency_s = time.perf_counter() - started
        response.raise_for_status()
        body = response.json()
        usage = body.get("usage", {})
        return {
            "latency_s": latency_s,
            "prompt_tokens": int(usage.get("prompt_tokens", 0) or 0),
            "completion_tokens": int(usage.get("completion_tokens", 0) or 0),
            "total_tokens": int(usage.get("total_tokens", 0) or 0),
        }

    started_all = time.perf_counter()
    results: list[dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(one_request, request_id) for request_id in range(num_requests)]
        for future in as_completed(futures):
            results.append(future.result())
    wall_time_s = time.perf_counter() - started_all

    latencies_ms = [item["latency_s"] * 1000.0 for item in results]
    prompt_tokens = sum(item["prompt_tokens"] for item in results)
    completion_tokens = sum(item["completion_tokens"] for item in results)
    total_tokens = sum(item["total_tokens"] for item in results)

    return {
        "model": model,
        "num_requests": num_requests,
        "concurrency": concurrency,
        "max_tokens": max_tokens,
        "completed_requests": len(results),
        "wall_time_s": wall_time_s,
        "request_throughput_rps": len(results) / wall_time_s if wall_time_s else None,
        "prompt_token_throughput_tps": prompt_tokens / wall_time_s if wall_time_s else None,
        "completion_token_throughput_tps": completion_tokens / wall_time_s if wall_time_s else None,
        "total_token_throughput_tps": total_tokens / wall_time_s if wall_time_s else None,
        "mean_latency_ms": statistics.mean(latencies_ms) if latencies_ms else None,
        "p50_latency_ms": percentile(latencies_ms, 0.50),
        "p95_latency_ms": percentile(latencies_ms, 0.95),
        "max_latency_ms": max(latencies_ms) if latencies_ms else None,
    }


def stop_process(handle: ManagedProcess | None, *, grace_seconds: int = 20) -> None:
    if handle is None:
        return
    process = handle.process
    if process.poll() is not None:
        return

    print(f"Stopping {handle.name} (pid={process.pid})")
    process.send_signal(signal.SIGINT if sys.platform != "win32" else signal.CTRL_BREAK_EVENT)
    try:
        process.wait(timeout=grace_seconds)
    except subprocess.TimeoutExpired:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)


def save_json(path: str | Path, body: dict[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(body, indent=2), encoding="utf-8")


In [ ]:
import os
from pathlib import Path

from vllm_free_colab_benchmark_helper import choose_free_colab_model, detect_gpu

os.environ.setdefault("HF_HOME", "/content/hf-cache")

gpu = detect_gpu()
print("Detected GPU:", gpu)

if gpu is None:
    raise RuntimeError(
        "No NVIDIA GPU was detected. Free Colab resource availability varies by session. "
        "Reconnect or switch the runtime type to GPU before using this notebook."
    )

FORCE_MODEL = None
MODEL, MODEL_NOTE = choose_free_colab_model(gpu, FORCE_MODEL)

GPU_MEMORY_UTILIZATION = 0.85 if gpu.memory_gb < 20 else 0.90
MAX_MODEL_LEN = 4096
MAX_NUM_SEQS = 4 if gpu.memory_gb < 20 else 8
NUM_REQUESTS = 24 if gpu.memory_gb < 20 else 48
CONCURRENCY = 2 if gpu.memory_gb < 20 else 4
MAX_TOKENS = 96
STARTUP_TIMEOUT_S = 1800
VLLM_PORT = 8000
LOG_DIR = Path("/content/vllm-free-logs")
RESULT_DIR = Path("/content/vllm-free-results")
ENV_ROOT = Path("/content/vllm-free-envs")

print("Selected model:", MODEL)
print("Selection note:", MODEL_NOTE)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Benchmark requests:", NUM_REQUESTS)
print("Benchmark concurrency:", CONCURRENCY)


In [ ]:
import os
import subprocess
from pathlib import Path


def ensure_uv_env(env_path: Path) -> Path:
    if not env_path.exists():
        subprocess.run(["uv", "venv", str(env_path)], check=True)
    return env_path / "bin" / "python"


ENV_ROOT.mkdir(parents=True, exist_ok=True)
VLLM_ENV = ENV_ROOT / "vllm"
vllm_python = ensure_uv_env(VLLM_ENV)

subprocess.run(
    ["uv", "pip", "install", "--python", str(vllm_python), "vllm", "requests", "pandas", "matplotlib"],
    check=True,
)

VLLM_BIN = str(VLLM_ENV / "bin" / "vllm")
VLLM_SITE_PACKAGES = subprocess.run(
    [str(vllm_python), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
vllm_version = subprocess.run([VLLM_BIN, "--version"], capture_output=True, text=True).stdout.strip()

print("vLLM version:", vllm_version)
print("vLLM site-packages:", VLLM_SITE_PACKAGES)


## Launch vLLM

This starts an OpenAI-compatible endpoint at `http://127.0.0.1:8000/v1`.


In [ ]:
from vllm_free_colab_benchmark_helper import (
    ensure_server_ready,
    start_logged_process,
    tail_log,
    warmup_chat,
)

server_env = os.environ.copy()
server_env["HF_HOME"] = os.environ["HF_HOME"]
server_env["VIRTUAL_ENV"] = str(VLLM_ENV)
server_env["PATH"] = f"{VLLM_ENV / 'bin'}:{server_env['PATH']}"
server_env["PYTHONNOUSERSITE"] = "1"
server_env["PYTHONPATH"] = VLLM_SITE_PACKAGES
server_env["MPLBACKEND"] = "Agg"

vllm_handle = start_logged_process(
    "vllm_server",
    [
        VLLM_BIN,
        "serve",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        str(VLLM_PORT),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--dtype",
        "half",
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--disable-log-requests",
    ],
    log_dir=LOG_DIR,
    env=server_env,
)

ensure_server_ready(
    f"http://127.0.0.1:{VLLM_PORT}",
    vllm_handle,
    timeout_s=STARTUP_TIMEOUT_S,
)
warmup_chat(f"http://127.0.0.1:{VLLM_PORT}", MODEL)
print(tail_log(vllm_handle.log_path, lines=40))


## Run the Benchmark

The benchmark sends concurrent chat-completions requests and computes throughput and latency from the returned usage stats.


In [ ]:
from vllm_free_colab_benchmark_helper import benchmark_chat_completions, save_json

result = benchmark_chat_completions(
    base_url=f"http://127.0.0.1:{VLLM_PORT}",
    model=MODEL,
    num_requests=NUM_REQUESTS,
    concurrency=CONCURRENCY,
    max_tokens=MAX_TOKENS,
)

result_path = RESULT_DIR / "vllm_free_colab_benchmark.json"
save_json(result_path, result)
print("Saved results to:", result_path)
result


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.DataFrame([result])
display(summary)

plot_data = pd.DataFrame(
    {
        "metric": [
            "request_throughput_rps",
            "completion_token_throughput_tps",
            "mean_latency_ms",
            "p95_latency_ms",
        ],
        "value": [
            result["request_throughput_rps"],
            result["completion_token_throughput_tps"],
            result["mean_latency_ms"],
            result["p95_latency_ms"],
        ],
    }
)

ax = plot_data.plot.bar(x="metric", y="value", legend=False, figsize=(10, 4), color="#2b6cb0")
ax.set_title("vLLM Free Colab Benchmark Summary")
ax.set_ylabel("Value")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
from vllm_free_colab_benchmark_helper import stop_process

stop_process(vllm_handle)


## Reading the Result

- `request_throughput_rps`: completed requests per second
- `completion_token_throughput_tps`: generated output tokens per second
- `mean_latency_ms`: average end-to-end request latency
- `p95_latency_ms`: tail latency for the slower requests

This is a good notebook for getting a **real free-Colab vLLM number**. It is not the right notebook for MAX comparison, TPU benchmarking, or reproducing the B200 Gemma 4 blog claim.
